# HeartLens AI — Colab training & experiments

Runtime: **T4 GPU** (Edit → Notebook settings → Hardware accelerator → GPU).

Run the cells **top to bottom**. Each experiment is its own cell, so one
failure no longer kills the rest — fix, re-run that single cell, continue.
Results land in `heart-lens-training/results/`; models in `heart-lens-training/models/`.


In [12]:
# 1. Get the code (fresh, non-nesting clone — safe to re-run)
%cd /content
!rm -rf heartlens
!git clone --depth 1 https://github.com/touhidsiddiqueeraj-bit/heartlens.git
%cd /content/heartlens

/content
Cloning into 'heartlens'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 73 (delta 0), reused 44 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 133.74 KiB | 1.94 MiB/s, done.
/content/heartlens


In [13]:
# 2. Dependencies (only the few not preinstalled in Colab)
!pip install -q wfdb fpdf2

# Optional: keep the session alive during the long training cells
from IPython.display import display, Javascript
display(Javascript('''
function ClickConnect(){ colab.gcloud.drive.notebooks._refClick(1); }
setInterval(ClickConnect, 60000);
'''))

<IPython.core.display.Javascript object>

In [14]:
# 3. Sanity: data split composition (expect all 3 classes in train/val/test)
%cd /content/heartlens/heart-lens-training
!python3 diagnose_data.py --data-dir ./mitdb 2>/dev/null | grep -E "bincount|records=|SKIP|nan"

/content/heartlens/heart-lens-training
train_classifier by_class[0] n=3000 records=40 ['100', '101', '102', '103', '104', '105', '106', '108', '112', '113', '114', '115', '116', '117', '119', '121', '122', '123', '200', '201', '202', '203', '205', '208', '209', '210', '212', '213', '215', '217', '219', '220', '221', '222', '223', '228', '230', '231', '233', '234']
train_classifier by_class[1] n=712 records=18 ['100', '101', '118', '124', '200', '201', '202', '205', '209', '213', '215', '219', '220', '222', '223', '231', '232', '233']
train_classifier by_class[2] n=2269 records=32 ['102', '104', '105', '106', '107', '108', '109', '111', '114', '116', '118', '119', '123', '124', '200', '201', '202', '203', '205', '207', '208', '210', '213', '214', '215', '217', '219', '221', '223', '228', '231', '233']
record_level_split y bincounts:
  X_tr: shape=(4251, 360) nan=0 min=-1.0000 max=1.0000
  X_va: shape=(812, 360) nan=0 min=-1.0000 max=1.0000
  X_te: shape=(918, 360) nan=0 min=-1.0000 max=

In [15]:
# 4. Train robust denoiser + 3-class classifier, int8 sizes, PDF report
%cd /content/heartlens
!python3 auto_train.py --epochs 30 --max-per-class 3000

/content/heartlens
2026-08-19 15:38:26.315359: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-19 15:38:26.389483: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
  HeartLens AI — Automated Training Pipeline
  Data:       ./mitdb
  Epochs:     30
  Max/class:  3000

  Loading dataset statistics...
Directory ./mitdb not found. Downloading...
Generating record list for: 100
Generating record list for: 101
Generating record list for: 102
Generating record list for: 103
Generating record list for: 104
Generating record list for: 105
Generating record list for: 106
Generating record list for: 107
Generating record list for: 108
Generating record list for: 109
Generating record lis

In [16]:
# Exp 1: grouped patient-level CV (5 folds x 3 seeds)
%cd /content/heartlens/heart-lens-training
!python3 group_kfold_eval.py --folds 5 --seeds 0,1,2 --epochs 30

/content/heartlens/heart-lens-training
2026-08-19 15:43:39.333873: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-19 15:43:39.408112: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
  100: 758 segments
  101: 643 segments
  102: 100 segments
  103: 703 segments
  104: 105 segments
  105: 832 segments
  106: 645 segments
  107: 4 segments
  108: 559 segments
  109: 11 segments
  111: 1 segments
  112: 853 segments
  113: 575 segments
  114: 552 segments
  115: 634 segments
  116: 796 segments
  117: 504 segments
  118: 35 segments
  119: 659 segments
  121: 607 segments
  122: 836 segments
  123: 504 segments
  124: 21 segments
  200: 869 segments
  201: 734 segments
  202: 5

In [17]:
# Exp 2: noise robustness Raw/Filter/AE x SNR (also trains + saves robust_*.keras)
%cd /content/heartlens/heart-lens-training
!python3 evaluate_noise_robustness.py --epochs 30

/content/heartlens/heart-lens-training
2026-08-19 16:26:40.527365: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-19 16:26:40.618416: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
  100: 758 segments
  101: 643 segments
  102: 100 segments
  103: 703 segments
  104: 105 segments
  105: 832 segments
  106: 645 segments
  107: 4 segments
  108: 559 segments
  109: 11 segments
  111: 1 segments
  112: 853 segments
  113: 575 segments
  114: 552 segments
  115: 634 segments
  116: 796 segments
  117: 504 segments
  118: 35 segments
  119: 659 segments
  121: 607 segments
  122: 836 segments
  123: 504 segments
  124: 21 segments
  200: 869 segments
  201: 734 segments
  202: 5

In [18]:
# Exp 3: external generalization mitdb -> SVDB + afdb
%cd /content/heartlens/heart-lens-training
!python3 external_validation.py --epochs 30

/content/heartlens/heart-lens-training
2026-08-19 16:33:50.102287: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-19 16:33:50.186007: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
  100: 758 segments
  101: 643 segments
  102: 100 segments
  103: 703 segments
  104: 105 segments
  105: 832 segments
  106: 645 segments
  107: 4 segments
  108: 559 segments
  109: 11 segments
  111: 1 segments
  112: 853 segments
  113: 575 segments
  114: 552 segments
  115: 634 segments
  116: 796 segments
  117: 504 segments
  118: 35 segments
  119: 659 segments
  121: 607 segments
  122: 836 segments
  123: 504 segments
  124: 21 segments
  200: 869 segments
  201: 734 segments
  202: 5

In [19]:
# Exp 4: FP32 vs INT8 delta per architecture (CNN/LSTM/GRU/TCN) + size
%cd /content/heartlens/heart-lens-training
!python3 compare_models.py --epochs 30

/content/heartlens/heart-lens-training
2026-08-19 16:41:59.541743: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-19 16:41:59.624148: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
  100: 758 segments
  101: 643 segments
  102: 100 segments
  103: 703 segments
  104: 105 segments
  105: 832 segments
  106: 645 segments
  107: 4 segments
  108: 559 segments
  109: 11 segments
  111: 1 segments
  112: 853 segments
  113: 575 segments
  114: 552 segments
  115: 634 segments
  116: 796 segments
  117: 504 segments
  118: 35 segments
  119: 659 segments
  121: 607 segments
  122: 836 segments
  123: 504 segments
  124: 21 segments
  200: 869 segments
  201: 734 segments
  202: 5

In [20]:
# Calibration: temperature scaling -> writes CALIB_TEMPERATURE to firmware Config.h
%cd /content/heartlens/heart-lens-training
!python3 calibrate.py --epochs 30 --write-config

/content/heartlens/heart-lens-training
2026-08-19 16:50:15.412377: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-19 16:50:15.553326: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
  100: 758 segments
  101: 643 segments
  102: 100 segments
  103: 703 segments
  104: 105 segments
  105: 832 segments
  106: 645 segments
  107: 4 segments
  108: 559 segments
  109: 11 segments
  111: 1 segments
  112: 853 segments
  113: 575 segments
  114: 552 segments
  115: 634 segments
  116: 796 segments
  117: 504 segments
  118: 35 segments
  119: 659 segments
  121: 607 segments
  122: 836 segments
  123: 504 segments
  124: 21 segments
  200: 869 segments
  201: 734 segments
  202: 5

In [21]:
# Export int8 firmware models (robust_classifier_int8.tflite + robust_denoiser_int8.tflite)
%cd /content/heartlens/heart-lens-training
!python3 export_firmware_models.py

/content/heartlens/heart-lens-training
2026-08-19 16:54:57.175766: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-19 16:54:57.297425: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
  100: 758 segments
  101: 643 segments
  102: 100 segments
  103: 703 segments
  104: 105 segments
  105: 832 segments
  106: 645 segments
  107: 4 segments
  108: 559 segments
  109: 11 segments
  111: 1 segments
  112: 853 segments
  113: 575 segments
  114: 552 segments
  115: 634 segments
  116: 796 segments
  117: 504 segments
  118: 35 segments
  119: 659 segments
  121: 607 segments
  122: 836 segments
  123: 504 segments
  124: 21 segments
  200: 869 segments
  201: 734 segments
  202: 5

In [22]:
# 5. Bundle results for download (models, results JSON, report PDF)
%cd /content/heartlens/heart-lens-training
!mkdir -p /content/out && cp -r models /content/out/ && cp -r results /content/out/
!cp ../auto_train_output/training_report.pdf /content/out/ 2>/dev/null; true
!cd /content/out && zip -qr heartlens_results.zip . && ls -lh heartlens_results.zip

/content/heartlens/heart-lens-training
-rw-r--r-- 1 root root 3.0M Aug 19 16:55 heartlens_results.zip


## After downloading `heartlens_results.zip`

Unzip into the local repo (the assistant wires the firmware afterwards):

```
mkdir -p heart-lens-training
unzip ~/heartlens_results.zip -d heart-lens-training/   # or wherever you saved it
```

Expected healthy results after the cap fix:
- `group_kfold.json` — all three per-class F1s > 0, macro ~0.8
- `noise_robustness.json` — raw/filter/autoencoder macro F1 rising with SNR
- `external_validation.json` — SVDB report present (record 813 skipped)
- `model_comparison.json` — 4 rows, LSTM/GRU int8 converted (no CudnnRNNV3 error)
